In [ ]:
import json
import google.generativeai as genai
from typing import Dict, List, Tuple
import time
import pandas as pd

# Configure Gemini
genai.configure(api_key='')
model = genai.GenerativeModel('gemini-2.0-flash')

In [38]:


EXTRACTION_PROMPT = """Analyze this story and extract symbolic narrative elements in JSON format.

Story: {story}

Extract the following elements:

1. **Abstract Theme**: The core ideas, motifs, moral lessons, or philosophical concepts (2-4 keywords)
2. **Key Events**: The main sequence of events/actions in chronological order (4-8 events as short phrases)
3. **Outcomes**: The final results or resolutions (2-3 outcome types)
4. **Character Arcs**: How characters change or what happens to them
5. **Conflict Type**: The nature of the central conflict

Return ONLY valid JSON in this exact format:
{{
    "themes": ["theme1", "theme2"],
    "events": ["event1", "event2", "event3"],
    "outcomes": ["outcome1", "outcome2"],
    "character_arcs": ["arc1", "arc2"],
    "conflict_type": "conflict_type"
}}

Be concise and focus on narrative structure."""

COMPARISON_PROMPT = """You are an expert in narrative analysis. Compare these stories based on NARRATIVE SIMILARITY.

Narrative similarity is defined by three core components:
1. **Abstract Theme**: The ideas, motifs, and moral lessons
2. **Course of Action**: The sequence of central events and turning points
3. **Outcomes**: The results and resolutions of the story

ANCHOR STORY:
Text: {anchor_text}

Symbolic Analysis:
- Themes: {anchor_themes}
- Key Events: {anchor_events}
- Outcomes: {anchor_outcomes}
- Character Arcs: {anchor_arcs}
- Conflict: {anchor_conflict}

---

OPTION A:
Text: {text_a}

Symbolic Analysis:
- Themes: {a_themes}
- Key Events: {a_events}
- Outcomes: {a_outcomes}
- Character Arcs: {a_arcs}
- Conflict: {a_conflict}

---

OPTION B:
Text: {text_b}

Symbolic Analysis:
- Themes: {b_themes}
- Key Events: {b_events}
- Outcomes: {b_outcomes}
- Character Arcs: {b_arcs}
- Conflict: {b_conflict}

---

INSTRUCTIONS:
1. Compare Option A vs Anchor on: themes, events sequence, and outcomes
2. Compare Option B vs Anchor on: themes, events sequence, and outcomes
3. Determine which option shares more narrative elements with the anchor

Think step by step about the three core components, then answer with ONLY "A" or "B".

Your answer: (ONLY A or B strictly, no explanations)"""

In [ ]:
def extract_symbolic_elements(story, max_retries= 3):
    """Extract symbolic narrative elements using Gemini."""
    for attempt in range(max_retries):
        try:
            prompt = EXTRACTION_PROMPT.format(story=story)
            response = model.generate_content(prompt)
            
            # Clean response
            text = response.text.strip()
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()
            
            elements = json.loads(text)
            
            # Validate structure
            required_keys = ["themes", "events", "outcomes"]
            if all(key in elements for key in required_keys):
                return elements
            else:
                print(f"Missing keys in response, attempt {attempt + 1}")
                
        except json.JSONDecodeError as e:
            print(f"JSON decode error on attempt {attempt + 1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    # Return empty structure if all retries fail
    return {
        "themes": [],
        "events": [],
        "outcomes": [],
        "character_arcs": [],
        "conflict_type": ""
    }


In [ ]:

def hybrid_comparison(anchor, text_a, text_b, anchor_elements, a_elements, b_elements, max_retries=5):
    """
    Use LLM to compare stories with symbolic elements as structured context.
    Returns True if text_a is closer, False if text_b is closer.
    """
    
    prompt = COMPARISON_PROMPT.format(
        anchor_text=anchor,
        anchor_themes=", ".join(anchor_elements.get("themes", [])),
        anchor_events=" → ".join(anchor_elements.get("events", [])),
        anchor_outcomes=", ".join(anchor_elements.get("outcomes", [])),
        anchor_arcs=", ".join(anchor_elements.get("character_arcs", [])),
        anchor_conflict=anchor_elements.get("conflict_type", ""),
        
        text_a=text_a,
        a_themes=", ".join(a_elements.get("themes", [])),
        a_events=" → ".join(a_elements.get("events", [])),
        a_outcomes=", ".join(a_elements.get("outcomes", [])),
        a_arcs=", ".join(a_elements.get("character_arcs", [])),
        a_conflict=a_elements.get("conflict_type", ""),
        
        text_b=text_b,
        b_themes=", ".join(b_elements.get("themes", [])),
        b_events=" → ".join(b_elements.get("events", [])),
        b_outcomes=", ".join(b_elements.get("outcomes", [])),
        b_arcs=", ".join(b_elements.get("character_arcs", [])),
        b_conflict=b_elements.get("conflict_type", "")
    )
    
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            
            # Parse answer
            if 'A' in answer and 'B' not in answer:
                return True
            elif 'B' in answer and 'A' not in answer:
                return False
            elif answer.startswith('A'):
                return True
            elif answer.startswith('B'):
                return False
            else:
                print(f"Ambiguous response on attempt {attempt + 1}: {answer}")
                time.sleep(1)
                
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    # Default to A if all retries fail
    print("All retries failed, defaulting to A")
    return True

In [ ]:


def process_triplet(anchor, text_a, text_b):
    """
    Process a triplet using hybrid symbolic-neural approach.
    Returns (text_a_is_closer, debug_info)
    """
    print("Extracting symbolic elements...")
    
    # Extract symbolic elements from all three stories
    anchor_elements = extract_symbolic_elements(anchor)
    a_elements = extract_symbolic_elements(text_a)
    b_elements = extract_symbolic_elements(text_b)
    
    print("Performing hybrid comparison...")
    
    # Use LLM with structured symbolic context
    text_a_is_closer = hybrid_comparison(
        anchor, text_a, text_b,
        anchor_elements, a_elements, b_elements
    )
    
    debug_info = {
        "anchor_elements": anchor_elements,
        "a_elements": a_elements,
        "b_elements": b_elements,
        "text_a_is_closer": text_a_is_closer
    }
    
    return text_a_is_closer, debug_info

In [ ]:


def evaluate_validation(data):
    """Evaluate on validation set with ground truth labels."""
    correct = 0
    total = len(data)
    
    for idx, item in enumerate(data):
        print(f"\n{'='*60}")
        print(f"Validation {idx + 1}/{total}")
        print(f"{'='*60}")
        
        try:
            text_a_is_closer, debug_info = process_triplet(
                item["anchor_text"],
                item["text_a"],
                item["text_b"]
            )
            
            ground_truth = item["text_a_is_closer"]
            is_correct = text_a_is_closer == ground_truth
            
            if is_correct:
                correct += 1
            
            # print(f"\nPrediction: {text_a_is_closer}")
            # print(f"Ground truth: {ground_truth}")
            # print(f"Result: {' CORRECT' if is_correct else ' WRONG'}")
            
            time.sleep(1)
            
        except Exception as e:
            print(f"Error: {e}")
    
    accuracy = correct / total if total > 0 else 0
    print(f"\n{'='*60}")
    print(f"VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    print(f"{'='*60}")
    return accuracy


val_data = pd.read_json('../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True)
val_data = val_data.to_dict(orient='records')
evaluate_validation(val_data)




VALIDATION RESULTS
Accuracy: 71.00% (142/200)
